# Did the deep model survive past 2016?

The CNN+Transformer of Guijarro-Ordonez, Pelger & Zanotti
([arXiv:2106.04028](https://arxiv.org/abs/2106.04028)) — **their model code, their
preprocessing** — trained on **our survivorship-free residuals**, 2002 → September
2026. Their own published residuals stop at 2016-12.

Training starts in 2002 deliberately. An earlier version of this notebook started
in 2012, which meant the first block learned from 2012–2015 — an era where the
pattern it needed to find had already mostly decayed. The paper's model trained on
1998–2001, when one-day reversal ran at Sharpe 3–6. Starting in 2002 gives the
network a comparable teacher and produces 20 test blocks spanning 2006–2026.

Non-neural arms on these identical residuals:

| arm | 2002–2008 | 2009–2016 | 2017–2026 |
|---|---|---|---|
| reversal L=1 | +2.81 | +0.85 | **−0.03** |
| reversal L=5 | +1.87 | +0.87 | +0.36 |
| **reversal L=30** | +0.86 | +0.74 | **+0.47** |
| L=30 turnover | 0.25 | 0.25 | 0.25 |

Fast reversal is dead; slow reversal halved and then stopped decaying. On their
1998–2016 residuals the network scored 4.92 gross against a permuted null of 0.46
— real, but earned almost entirely before 2009 and beaten by L=30 after costs.

**Runtime** ~55 min for the real arm (20 blocks), same again per null.

In [ ]:
#@title 1. Load the residuals (upload cache/mp_eps_cnn_full.npz -- 21 MB)
# From the repo:  python3 src/mp_export_cnn.py 2002-01-01 2026-09-18 \
#                        cache/mp_eps_K5_N900.parquet cache/mp_eps_cnn_full
import os, numpy as np, pandas as pd
F = "mp_eps_cnn_full.npz"
if not os.path.exists(F):
    print(f"{F} not found -- opening an upload dialog.")
    try:
        from google.colab import files; files.upload()
    except Exception as e: print("upload widget unavailable:", e)
if not os.path.exists(F):
    raise SystemExit(f"\n{F} still missing; nothing below will run without it.")
z = np.load(F, allow_pickle=True)
data  = z["eps"].astype(np.float32)
dates = pd.DatetimeIndex(z["dates"])
print(f"residuals {data.shape}  {dates[0].date()} .. {dates[-1].date()}")
print(f"active names/day: median {np.median((data != 0).sum(1)):.0f}")


In [ ]:
#@title 2. Their model and preprocessing (trimmed clone, ~2 MB of code)
import torch, time, json, subprocess
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available()
      else "NONE -- Runtime > Change runtime type > GPU")
def sh(c): return subprocess.run(c, shell=True, capture_output=True, text=True).returncode
if not os.path.isdir("dlsa-public"):
    sh("git clone --depth 1 --filter=blob:none --no-checkout -q "
       "https://github.com/gregzanotti/dlsa-public.git")
    sh("cd dlsa-public && git sparse-checkout set --no-cone '/*' '!/residuals/*' "
       "&& git checkout -q main")
import sys; sys.path.insert(0, "dlsa-public")
from preprocess import preprocess_cumsum          # THEIR code
from models.CNNTransformer import CNNTransformer  # THEIR code
print("imported their preprocess_cumsum and CNNTransformer")

In [ ]:
#@title 3. Parameters
LOOKBACK     = 30    #@param {type:"integer"}
EPOCHS       = 30    #@param {type:"integer"}   # their config: 100
RETRAIN_FREQ = 250   #@param {type:"integer"}   # their config: 125
TRAIN_LEN    = 1000  #@param {type:"integer"}
BATCH_DAYS   = 125   #@param {type:"integer"}
LR           = 0.001
N_NULL       = 1     #@param {type:"integer"}
BORROW_BP_YR = 35.0

windows, idxs = preprocess_cumsum(data, LOOKBACK)
idxs = idxs.numpy() if hasattr(idxs, "numpy") else idxs
wdates = dates[LOOKBACK:]
print("windows", windows.shape, " valid (day,stock) pairs", idxs.sum())

In [ ]:
#@title 4. Benchmarks on the SAME residuals, by era -- the bar to clear
def sharpe(r): return float(r.mean()/r.std()*np.sqrt(252)) if len(r) > 20 else float("nan")
BORROW_BP_YR = 35.0

def bench(Wt, sel, d, lb=LOOKBACK):
    w = np.where(sel, Wt, 0.0); w = w/np.abs(w).sum(1, keepdims=True).clip(1e-12)
    g  = np.array([float(w[i] @ d[lb+i]) for i in range(len(w))])
    to = np.abs(np.diff(w, axis=0)).sum(1); to = np.r_[to[0], to]
    bo = np.abs(np.clip(w, None, 0)).sum(1)*BORROW_BP_YR*1e-4/252
    return g, to, bo

last = windows[:, :, -1]
def rev(L): return -(last - (windows[:, :, -L-1] if L < LOOKBACK else 0.0))
ERAS = (("2006-2008","2006-01-01","2008-12-31"),
        ("2009-2016","2009-01-01","2016-12-31"),
        ("2017-2026","2017-01-01","2026-12-31"))
print("{:10s} {:>10s} {:>8s} {:>8s} {:>8s}".format("arm","era","gross","turn","net@1bp"))
print("-"*48)
for nm, Wt in (("L=1", rev(1)), ("L=5", rev(5)), ("L=30", rev(30))):
    g, to, bo = bench(Wt, idxs, data)
    for lab, lo, hi in ERAS:
        m = (wdates >= lo) & (wdates <= hi)
        print("{:10s} {:>10s} {:+8.2f} {:8.2f} {:+8.2f}".format(
              nm, lab, sharpe(g[m]), to[m].mean(), sharpe(g[m]-1e-4*to[m]-bo[m])))


In [ ]:
#@title 5. Training loop -- their model, per-block, turnover tracked
MAX_ROWS = 32768
def fwd(model, x):
    if x.shape[0] <= MAX_ROWS: return model(x).squeeze(-1)
    return torch.cat([model(x[i:i+MAX_ROWS]).squeeze(-1)
                      for i in range(0, x.shape[0], MAX_ROWS)])

def train_eval(d, sel_all, win_all, seed=0, null=None):
    dev = "cuda" if torch.cuda.is_available() else "cpu"
    torch.manual_seed(seed); np.random.seed(seed)
    rp = np.random.default_rng(seed + 7919)
    Tn, N = win_all.shape[0], sel_all.shape[1]
    dd = d
    if null == "permute":                 # destroy own-history -> own-future only
        dd = d.copy()
        for t in range(LOOKBACK, len(dd)):
            act = np.nonzero(dd[t])[0]
            if len(act) > 1: dd[t, act] = dd[t, act][rp.permutation(len(act))]
    blocks, all_r, all_to, all_d = [], [], [], []
    prev_w = np.zeros(N)
    starts = list(range(TRAIN_LEN, Tn - RETRAIN_FREQ, RETRAIN_FREQ))
    for bi, s0 in enumerate(starts):
        model = CNNTransformer(logdir=None, random_seed=seed, lookback=LOOKBACK,
                               device=dev, dropout=0.25, filter_numbers=[1,8],
                               filter_size=2, attention_heads=4,
                               hidden_units_factor=2, normalization_conv=True,
                               use_transformer=True, use_convolution=True).to(dev)
        opt = torch.optim.Adam(model.parameters(), lr=LR)
        tr = np.arange(s0-TRAIN_LEN, s0)
        for ep in range(EPOCHS):
            for ch in [tr[i:i+BATCH_DAYS] for i in range(0, len(tr), BATCH_DAYS)]:
                sub = sel_all[ch]
                x = torch.tensor(win_all[ch][sub], device=dev)
                if len(x) < 10: continue
                W = torch.zeros(len(ch), N, device=dev)
                W[torch.tensor(sub, device=dev)] = fwd(model, x)
                W = W / W.abs().sum(1, keepdim=True).clamp(min=1e-9)
                r = (W * torch.tensor(dd[LOOKBACK+ch], device=dev)).sum(1)
                loss = -r.mean()/r.std().clamp(min=1e-9)
                opt.zero_grad(); loss.backward(); opt.step()
        model.eval(); br, bt = [], []
        te = np.arange(s0, min(s0+RETRAIN_FREQ, Tn))
        with torch.no_grad():
            for ch in [te[i:i+BATCH_DAYS] for i in range(0, len(te), BATCH_DAYS)]:
                sub = sel_all[ch]
                x = torch.tensor(win_all[ch][sub], device=dev)
                if len(x) < 10: continue
                W = torch.zeros(len(ch), N, device=dev)
                W[torch.tensor(sub, device=dev)] = fwd(model, x)
                W = W / W.abs().sum(1, keepdim=True).clamp(min=1e-9)
                br.append((W*torch.tensor(dd[LOOKBACK+ch], device=dev)).sum(1).cpu().numpy())
                Wc = W.cpu().numpy().astype(np.float64)
                for k in range(len(Wc)):
                    bt.append(np.abs(Wc[k]-prev_w).sum()); prev_w = Wc[k]
        br = np.concatenate(br); bt = np.array(bt)
        all_r.append(br); all_to.append(bt); all_d.append(wdates[te][:len(br)])
        blocks.append({"year": str(wdates[s0].date()), "sharpe": sharpe(br),
                       "turnover": float(bt.mean())})
        print(f"    block {bi+1}/{len(starts)}  {blocks[-1]['year']}  "
              f"SR {blocks[-1]['sharpe']:+.2f}  turn {blocks[-1]['turnover']:.2f}"
              f"   ({time.time()-T0:.0f}s)", flush=True)
    return (np.concatenate(all_r), np.concatenate(all_to),
            pd.DatetimeIndex(np.concatenate(all_d)), blocks)

In [ ]:
#@title 6. Run -- real arm, then the permuted null
T0 = time.time()
print("REAL arm (20 blocks, ~55 min) ...", flush=True)
r, to, dd_, blocks = train_eval(data, idxs, windows, seed=0)
bo = np.zeros_like(to)   # borrow is applied per-era below from the weight path

def era_stats(g, t, d_, lab):
    m = {"2006-2008": (d_ >= "2006-01-01") & (d_ <= "2008-12-31"),
         "2009-2016": (d_ >= "2009-01-01") & (d_ <= "2016-12-31"),
         "2017-2026": (d_ >= "2017-01-01")}[lab]
    lo, hi = 0.0, 500.0
    for _ in range(60):
        mm = (lo+hi)/2
        if sharpe(g[m] - mm*1e-4*t[m]) > 0: lo = mm
        else: hi = mm
    return dict(gross=sharpe(g[m]), turn=float(t[m].mean()), be=lo,
                net1=sharpe(g[m]-1e-4*t[m]))

res = {"blocks": blocks, "gross_all": sharpe(r)}
print("\n{:12s} {:>8s} {:>7s} {:>8s} {:>9s}".format("CNN era","gross","turn","b/e bp","net@1bp"))
print("-"*48)
for lab in ("2006-2008","2009-2016","2017-2026"):
    s = era_stats(r, to, dd_, lab); res[lab] = s
    print("{:12s} {:+8.2f} {:7.2f} {:8.1f} {:+9.2f}".format(lab, s["gross"], s["turn"], s["be"], s["net1"]))

nulls = []
for i in range(N_NULL):
    print(f"\nNULL [permute] {i+1}/{N_NULL} ...", flush=True)
    rn, tn, dn, bn = train_eval(data, idxs, windows, seed=100+i, null="permute")
    nulls.append({lab: era_stats(rn, tn, dn, lab)["gross"]
                  for lab in ("2006-2008","2009-2016","2017-2026")})
    print("  null gross by era:", {k: round(v,2) for k,v in nulls[-1].items()})
res["null"] = nulls
json.dump(res, open("modern_results.json","w"), indent=2)
np.save("cnn_modern_daily.npy", np.vstack([r, to]))

print("\n" + "="*62)
print("{:22s} {:>12s} {:>12s} {:>12s}".format("2017-2026","CNN","null","reversal L=30"))
print("-"*62)
nm17 = np.mean([n["2017-2026"] for n in nulls]) if nulls else float("nan")
print("{:22s} {:+12.2f} {:+12.2f} {:+12.2f}".format("gross", res["2017-2026"]["gross"], nm17, 0.47))
print("{:22s} {:12.2f} {:>12s} {:12.2f}".format("turnover", res["2017-2026"]["turn"], "-", 0.25))
print("{:22s} {:+12.2f} {:>12s} {:+12.2f}".format("net @1bp", res["2017-2026"]["net1"], "-", 0.37))
print("="*62)
try:
    from google.colab import files
    for f in ("modern_results.json","cnn_modern_daily.npy"): files.download(f)
except Exception as e: print("download skipped:", e)
